# 🔍 Análise Exploratória de Dados (EDA) e Modelagem

Este notebook realiza a análise dos dados de consumo de energia e emissões de CO2, focando em:
1. Limpeza e preparação dos dados.
2. Visualização de tendências setoriais e regionais.
3. Classificação de impacto ambiental.
4. Treinamento de um modelo de Árvore de Decisão.

---

In [ ]:
# Folder creation commented out to preserve local project structure
# import os
# 
# # Define o nome da pasta principal
# nome_projeto = "Analise_Pegada_Carbono"
# 
# # Define as subpastas em português
# subpastas = [
#     "dados/brutos",
#     "dados/processados",
#     "notebooks",
#     "docs",
#     "scripts"
# ]
# 
# # Cria a estrutura
# if not os.path.exists(nome_projeto):
#     os.makedirs(nome_projeto)
#     print(f"✅ Pasta raiz '{nome_projeto}' criada.")
# 
# for pasta in subpastas:
#     caminho = os.path.join(nome_projeto, pasta)
#     if not os.path.exists(caminho):
#         os.makedirs(caminho)
#         print(f"  + Subpasta '{pasta}' criada.")
# 
# print("\n🚀 Estrutura pronta para começar o projeto!")

In [ ]:
import pandas as pd

# 1. Definir o caminho para o dataset sintético gerado
caminho_dados = 'https://raw.githubusercontent.com/carbon-footprint-analysis/carbon-footprint-analysis/main/data/processed/synthetic_energy_emissions_dataset.csv'

# 2. Carregar o dataset usando o pandas
# O dataset sintético possui um índice na primeira coluna que podemos ignorar
df = pd.read_csv(caminho_dados).drop(columns=['Unnamed: 0'], errors='ignore')

# 3. Inspeção Visual
print("✅ Dataset carregado com sucesso!")
print(f"O arquivo contém {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

print("--- Primeiras 5 linhas do Dataset ---")
display(df.head())

print("\n--- Resumo Técnico (Tipos de Dados e Nulos) ---")
df.info()

print("\n--- Estatísticas Descritivas Básicas ---")
display(df.describe())

In [ ]:
# 1. Converter a coluna de data para o formato correto de data
df['data'] = pd.to_datetime(df['data'])

# 2. Converter colunas numéricas
cols_numericas = ['consumo_kwh', 'emissao_co2']

for col in cols_numericas:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Verificar se a conversão deu certo
print("✅ Conversão concluída!")
print("\n--- Novos Tipos de Dados ---")
print(df.dtypes)

print("\n--- Estatísticas Atualizadas ---")
display(df.describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Silencia avisos genéricos
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. PREPARAÇÃO DOS DADOS PARA O RELATÓRIO ---
# Emissões por Setor
setor_emissao = df.groupby('setor')['emissao_co2'].sum().sort_values(ascending=False).reset_index()

# Emissões por Estado (Top 5 para o console)
estado_emissao_full = df.groupby('estado')['emissao_co2'].sum().sort_values(ascending=False).reset_index()
top_estados = estado_emissao_full.head(10)

# Intensidade de Carbono
resumo_fontes = df.groupby('fonte_energia')[['emissao_co2', 'consumo_kwh']].sum()
intensidade = (resumo_fontes['emissao_co2'] / resumo_fontes['consumo_kwh']).sort_values(ascending=False)

# --- 2. RELATÓRIO POR ESCRITO NO CONSOLE ---
print("="*50)
print("📊 RELATÓRIO EXECUTIVO DE EMISSÕES DE CO2")
print("="*50)

print("\n[1] EMISSÕES TOTAIS POR SETOR (kg CO2):")
for index, row in setor_emissao.iterrows():
    print(f" - {row['setor'].capitalize()}: {row['emissao_co2']:,.2f} kg")

print("\n[2] TOP 5 ESTADOS COM MAIORES EMISSÕES:")
for index, row in estado_emissao_full.head(5).iterrows():
    print(f" - {row['estado']}: {row['emissao_co2']:,.2f} kg")

print("\n[3] INTENSIDADE DE CARBONO POR FONTE (kg CO2/kWh):")
for fonte, valor in intensidade.items():
    print(f" - {fonte.capitalize()}: {valor:.4f} kg CO2/kWh")

print("\n" + "="*50)

# --- 3. PARTE VISUAL (GRÁFICOS) ---
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 10))

# Gráfico de Barras: Setor
plt.subplot(2, 2, 1)
sns.barplot(data=setor_emissao, x='setor', y='emissao_co2', hue='setor', palette='viridis', legend=False)
plt.title('Total de Emissões por Setor (kg CO2)')
plt.xticks(rotation=45)

# Gráfico de Barras: Estado
plt.subplot(2, 2, 2)
sns.barplot(data=top_estados, x='estado', y='emissao_co2', hue='estado', palette='magma', legend=False)
plt.title('Top 10 Estados com Maiores Emissões')

# Boxplot: Fonte de Energia
plt.subplot(2, 1, 2)
sns.boxplot(data=df, x='fonte_energia', y='emissao_co2', hue='fonte_energia', palette='Set2', legend=False)
plt.yscale('log')
plt.title('Distribuição de Emissões por Fonte de Energia (Escala Log)')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# 1. Criar a classificação baseada nos percentis das emissões
# Vamos dividir em 3 grupos iguais (33% cada)
limites = df['emissao_co2'].quantile([0.33, 0.66]).values

def categorizar_impacto(valor):
    if valor <= limites[0]:
        return 'Baixo Impacto'
    elif valor <= limites[1]:
        return 'Médio Impacto'
    else:
        return 'Alto Impacto'

# Aplicar a função
df['impacto_ambiental'] = df['emissao_co2'].apply(categorizar_impacto)

# 2. RELATÓRIO TEXTUAL DA NOVA COLUNA
print("="*50)
print("✅ NOVA VARIÁVEL: IMPACTO AMBIENTAL")
print("="*50)

contagem = df['impacto_ambiental'].value_counts()
porcentagem = df['impacto_ambiental'].value_counts(normalize=True) * 100

for cat in ['Baixo Impacto', 'Médio Impacto', 'Alto Impacto']:
    print(f" - {cat}: {contagem[cat]} registros ({porcentagem[cat]:.1f}%)")

print("\n" + "="*50)

# 3. SALVAR O DATASET PROCESSADO
caminho_processado = '../data/processed/dados_energia_limpos.csv'
df.to_csv(caminho_processado, index=False)
print(f"💾 Arquivo salvo com sucesso em:\n{caminho_processado}")

# 4. VISUALIZAÇÃO DA DISTRIBUIÇÃO
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='impacto_ambiental', order=['Baixo Impacto', 'Médio Impacto', 'Alto Impacto'],
              hue='impacto_ambiental', palette='RdYlGn_r', legend=False)
plt.title('Distribuição das Categorias de Impacto Ambiental')
plt.ylabel('Quantidade de Registros')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix

# 1. PREPARAÇÃO DOS DADOS
# Vamos usar o consumo e o setor para prever o impacto ambiental
# Precisamos transformar o texto (setor) em números para o modelo entender
df_modelo = df.copy()
df_modelo['setor_encoded'] = df_modelo['setor'].astype('category').cat.codes

X = df_modelo[['consumo_kwh', 'setor_encoded']]
y = df_modelo['impacto_ambiental']

# Dividir em Treino (80%) e Teste (20%)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. TREINAMENTO DO MODELO
# Limitamos a profundidade (max_depth) para a árvore não ficar gigante e ser legível
modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_treino, y_treino)

# 3. RELATÓRIO DE DESEMPENHO NO CONSOLE
print("="*50)
print("🤖 Relatório do Modelo Preditivo")
print("="*50)
previsoes = modelo.predict(X_teste)
print(classification_report(y_teste, previsoes))

# 4. VISUALIZAÇÃO DA ÁRVORE (O "MAPA" DA DECISÃO)
plt.figure(figsize=(20,10))
plot_tree(modelo, feature_names=['Consumo (kWh)', 'Setor'],
          class_names=modelo.classes_, filled=True, rounded=True, fontsize=12)
plt.title("Árvore de Decisão: Classificação de Impacto Ambiental")
plt.show()